# 04 — Candidate Generation & Oracle Analysis

Generate N=32 candidates per question, execute & fingerprint, compute oracle metrics.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/SLM-PRJ')
os.chdir('/kaggle/working/SLM-PRJ')

from pathlib import Path
import json

## Generate Candidates (GPU required)

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_schema
from research.candidate_generator import generate_candidates, save_candidates

DATA_DIR = Path('/kaggle/input/bird-benchmark')
MODEL_PATH = 'results/lora/best_adapter/merged'  # adjust to your best model
OUTPUT = Path('results/candidates/candidates_dev.jsonl')

bird_dev = load_bird_dev(DATA_DIR)
prompts = []
for r in bird_dev:
    schema = get_schema(DATA_DIR, r['db_id'])
    prompts.append({**r, 'schema': schema})

print(f'Generating 32 candidates for {len(prompts)} questions...')
results = generate_candidates(MODEL_PATH, prompts, n_candidates=32)
save_candidates(results, OUTPUT)
print(f'Saved to {OUTPUT}')

## Execute & Fingerprint

In [ ]:
from research.exec_runner import execute_candidates
from benchmarks.bird_loader import get_db_path

EXEC_OUTPUT = Path('results/candidates/exec_dev.jsonl')
stats = execute_candidates(
    'results/candidates/candidates_dev.jsonl',
    lambda db_id: get_db_path(DATA_DIR, db_id),
    EXEC_OUTPUT,
)
print(json.dumps(stats, indent=2))

## Oracle Analysis

In [ ]:
from research.oracle_analysis import load_exec_results, pass_at_k, sc_at_k, top2_coverage_at_k, analyze_by_group

records = load_exec_results(EXEC_OUTPUT)
db_path_fn = lambda db_id: get_db_path(DATA_DIR, db_id)

print('=== Oracle Metrics ===')
for k in [4, 8, 16, 32]:
    p = pass_at_k(records, db_path_fn, k)
    s = sc_at_k(records, db_path_fn, k)
    t = top2_coverage_at_k(records, db_path_fn, k)
    print(f'k={k:2d}  pass@k={p:.4f}  SC@k={s:.4f}  top2@k={t:.4f}')

In [ ]:
results_by_diff = analyze_by_group(records, db_path_fn, [4, 8, 16, 32], 'difficulty')
for group, metrics in results_by_diff.items():
    print(f'\n{group} (n={metrics["count"]})')
    for k, v in sorted(metrics.items()):
        if k != 'count':
            print(f'  {k}: {v}')

## Calibration Analysis

In [ ]:
from research.calibration import compute_calibration_data, rank_features

calib_data = compute_calibration_data(records, db_path_fn)
print(f'Calibration rows: {len(calib_data)}')

print('\n=== Feature AUC Ranking ===')
for name, auc in rank_features(calib_data):
    print(f'  {name:20s}  AUC={auc:.4f}')